## AnalystLab Africa Internship Week 5: Advanced Machine Learning





## Introduction

This week focuses on exploring advanced machine learning algorithms and improving predictive performance through model comparison and hyperparameter tuning.

Using the House Prices dataset from Week 1, I will prepare the data, establish a Linear Regression baseline, and compare its performance with more advanced regression models, including Decision Tree, Random Forest, and Gradient Boosting.

I will also use hyperparameter tuning to optimize one of the models and compare its performance before and after tuning. The models will be evaluated using appropriate regression metrics, including MAE, MSE, RMSE, and R² Score.

The main goal is to identify the best-performing model and understand how model selection and optimization can improve predictive performance.

In [1]:
# Data Manipulation, Visualisation, and Preprocessing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Model Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

sns.set_style("whitegrid")

In [2]:
# Load the House Prices dataset
house_prices = pd.read_csv("Housing.csv")

# Displaying the first five rows
house_prices.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [4]:
house_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [5]:
house_prices.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


### Dataset Structure

The House Prices dataset contains 545 observations and 13 variables. The target variable is `price`, while the remaining 12 variables represent different characteristics of the houses. The dataset contains 6 numerical variables and 7 categorical variables. All columns have 545 non-null values, indicating that there are no missing values that require treatment.

In [6]:
# Checking for duplicate rows
house_prices.duplicated().sum()

0

In [7]:
# No duplicate records were found, so no rows were removed.

In [8]:
# Check for missing values
house_prices.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [9]:
#The dataset contains no missing values

### Encoding Categorical Variables

In [10]:
# Encoding binary categorical variables
le = LabelEncoder()

binary_columns = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea"
]

for col in binary_columns:
    house_prices[col] = le.fit_transform(house_prices[col])

In [11]:
# One-hot encoding furnishing status
house_prices = pd.get_dummies(
    house_prices,
    columns=["furnishingstatus"],
    drop_first=True,
    dtype=int
)

In [12]:
house_prices.head() 

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0,0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0,0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1,0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0,0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0,0


In [13]:
house_prices.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   price                            545 non-null    int64
 1   area                             545 non-null    int64
 2   bedrooms                         545 non-null    int64
 3   bathrooms                        545 non-null    int64
 4   stories                          545 non-null    int64
 5   mainroad                         545 non-null    int32
 6   guestroom                        545 non-null    int32
 7   basement                         545 non-null    int32
 8   hotwaterheating                  545 non-null    int32
 9   airconditioning                  545 non-null    int32
 10  parking                          545 non-null    int64
 11  prefarea                         545 non-null    int32
 12  furnishingstatus_semi-furnished  545 non-null    i

In [15]:
# Separating the features and target variable
X = house_prices.drop("price", axis=1)
y = house_prices["price"]

In [16]:
# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Feature Scaling

Feature scaling was considered during preprocessing. However, scaling was not applied because the models used in this task, including Linear Regression, Decision Tree, Random Forest, and Gradient Boosting, can operate effectively without standardized features. Tree-based models in particular are not sensitive to differences in feature scale.

### Building a baseline model

Linear Regression

In [19]:
# Creating the Linear Regression model
baseline_model = LinearRegression()

In [20]:
# Training the model using the training data
baseline_model.fit(X_train, y_train)

LinearRegression()

In [22]:
# Making predictions on the test data
y_pred_baseline = baseline_model.predict(X_test)

In [31]:
# Calculating evaluation metrics
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)

mse_baseline = mean_squared_error(y_test, y_pred_baseline)

rmse_baseline = np.sqrt(mse_baseline)

r2_baseline = r2_score(y_test, y_pred_baseline)

print("Baseline Linear Regression Results")
print("MAE:", mae_baseline)
print("MSE:", mse_baseline)
print("RMSE:", rmse_baseline)
print("R² Score:", r2_baseline)

Baseline Linear Regression Results
MAE: 970043.403920164
MSE: 1754318687330.664
RMSE: 1324506.9600914388
R² Score: 0.6529242642153184


### Baseline Model Interpretation

The Linear Regression model achieved an RMSE of approximately ₦1.32 million and an R² score of 0.6529. This means that the model's predictions differ from the actual house prices by about ₦1.32 million on average, while explaining approximately 65.3% of the variation in house prices.

These results serve as the baseline for comparing the performance of the more advanced machine learning models. An improved model should ideally produce a lower RMSE and a higher R² score.

### Training the Advanced Machine Learning Models

#### Decision Tree

In [24]:
# Creating the Decision Tree Regressor
dt_model = DecisionTreeRegressor(random_state=42)

In [26]:
# Training the Decision Tree model
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [27]:
# Making predictions on the test data
y_pred_dt = dt_model.predict(X_test)

In [32]:
# Evaluating the Decision Tree model
mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree Results")
print("MAE:", mae_dt)
print("MSE:", mse_dt)
print("RMSE:", rmse_dt)
print("R² Score:", r2_dt)

Decision Tree Results
MAE: 1195266.0550458715
MSE: 2642802637614.6787
RMSE: 1625669.904259373
R² Score: 0.4771459275854347


### Decision Tree Interpretation

The Decision Tree Regressor achieved an RMSE of approximately ₦1.63 million and an R² score of 0.4771. Compared with the Linear Regression baseline, the Decision Tree produced a higher prediction error and explained less variation in house prices.

This suggests that the default Decision Tree configuration did not perform as well as Linear Regression on this dataset. The model may also be sensitive to the training data and could benefit from hyperparameter tuning.

#### Random Forest Regressor

In [30]:
# Creating the Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [33]:
# Training the Random Forest model
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [34]:
# Making predictions on the test data
y_pred_rf = rf_model.predict(X_test)

In [35]:
# Evaluating the Random Forest model
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

Random Forest Results
MAE: 1022560.0527522935
MSE: 1964193399645.3335
RMSE: 1401496.8425384816
R² Score: 0.6114024924156645


### Random Forest Interpretation

The Random Forest Regressor achieved an RMSE of approximately ₦1.40 million and an R² score of 0.6114. It performed considerably better than the individual Decision Tree, with lower prediction errors and a higher R² score.

However, Random Forest still performed slightly worse than the Linear Regression baseline, which achieved an RMSE of approximately ₦1.32 million and an R² score of 0.6529. This shows that a more complex model does not necessarily provide better predictions for this dataset.

#### Gradient Boosting Regressor

In [36]:
# Creating the Gradient Boosting Regressor
gb_model = GradientBoostingRegressor(random_state=42)

In [37]:
# Training the Gradient Boosting model
gb_model.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [38]:
# Making predictions on the test data
y_pred_gb = gb_model.predict(X_test)

In [39]:
# Evaluating the Gradient Boosting model
mae_gb = mean_absolute_error(y_test, y_pred_gb)

mse_gb = mean_squared_error(y_test, y_pred_gb)

rmse_gb = np.sqrt(mse_gb)

r2_gb = r2_score(y_test, y_pred_gb)

print("Gradient Boosting Results")
print("MAE:", mae_gb)
print("MSE:", mse_gb)
print("RMSE:", rmse_gb)
print("R² Score:", r2_gb)

Gradient Boosting Results
MAE: 960578.7795232662
MSE: 1689379037287.4016
RMSE: 1299761.146244725
R² Score: 0.6657719736897354


### Gradient Boosting Interpretation

The Gradient Boosting Regressor achieved an RMSE of approximately ₦1.30 million and an R² score of 0.6658. This makes it the best-performing model so far, outperforming the Linear Regression baseline with lower MAE and RMSE and a higher R² score.

The model explains approximately 66.6% of the variation in house prices. Its improved performance suggests that Gradient Boosting was better able to capture non-linear relationships and interactions among the housing features.

## HyperParameter Tuning

In [40]:
# Defining the hyperparameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10]
}

In [46]:
# Creating the Gradient Boosting model
gb_tuning = GradientBoostingRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=gb_tuning,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [47]:
# Fitting GridSearchCV on the training data
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [2, 3, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_root_mean_squared_error')

In [49]:
# Displaying the best hyperparameters
print("Best Parameters:")
print(grid_search.best_params_) 

Best Parameters:
{'learning_rate': 0.05, 'max_depth': 2, 'min_samples_split': 2, 'n_estimators': 100}


In [50]:
# Displaying the best cross-validation score
print("Best CV Score:", grid_search.best_score_)

Best CV Score: -1081553.1217116946


In [51]:
# Getting the best-performing model
tuned_gb_model = grid_search.best_estimator_

In [52]:
# Making predictions using the tuned model
y_pred_tuned = tuned_gb_model.predict(X_test)

In [53]:
# Evaluating the tuned Gradient Boosting model
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)

mse_tuned = mean_squared_error(y_test, y_pred_tuned)

rmse_tuned = np.sqrt(mse_tuned)

r2_tuned = r2_score(y_test, y_pred_tuned)

print("Tuned Gradient Boosting Results")
print("MAE:", mae_tuned)
print("MSE:", mse_tuned)
print("RMSE:", rmse_tuned)
print("R² Score:", r2_tuned)

Tuned Gradient Boosting Results
MAE: 1021988.1958462002
MSE: 1937963730283.183
RMSE: 1392107.657576519
R² Score: 0.6165917900381558


### Hyperparameter Tuning Interpretation

GridSearchCV identified the following parameters as the best combination based on 5-fold cross-validation: a learning rate of 0.05, maximum tree depth of 2, minimum samples split of 2, and 100 estimators.

However, the tuned model did not outperform the original Gradient Boosting model on the test set. The tuned model achieved an RMSE of approximately ₦1.39 million and an R² score of 0.6166, compared with ₦1.30 million and 0.6658 respectively for the original model.

This shows that hyperparameter tuning does not always guarantee improved test-set performance. In this case, the original Gradient Boosting configuration generalized better to the unseen test data.

## Model Evaluation & Performance

In [60]:
# Creating a comparison table for all models
model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "Tuned Gradient Boosting"
    ],
    "MAE Results": [
        mae_baseline,
        mae_dt,
        mae_rf,
        mae_gb,
        mae_tuned
    ],
    "MSE Results": [
        mse_baseline,
        mse_dt,
        mse_rf,
        mse_gb,
        mse_tuned
    ],
    "RMSE Results": [
        rmse_baseline,
        rmse_dt,
        rmse_rf,
        rmse_gb,
        rmse_tuned
    ],
    "R² Score Results": [
        r2_baseline,
        r2_dt,
        r2_rf,
        r2_gb,
        r2_tuned
    ]
})

model_comparison

,Model,MAE Results,MSE Results,RMSE Results,R² Score Results
0,Linear Regression,9.700434e+05,1.754319e+12,1.324507e+06,0.652924
1,Decision Tree,1.195266e+06,2.642803e+12,1.625670e+06,0.477146
2,Random Forest,1.022560e+06,1.964193e+12,1.401497e+06,0.611402
3,Gradient Boosting,9.605788e+05,1.689379e+12,1.299761e+06,0.665772
4,Tuned Gradient Boosting,1.021988e+06,1.937964e+12,1.392108e+06,0.616592


In [66]:
# Formatting the model comparison table for readability
model_comparison_formatted = model_comparison.copy()

model_comparison_formatted["MAE Results"] = (
    model_comparison_formatted["MAE Results"].map(lambda x: f"{x:,.2f}")
)

model_comparison_formatted["MSE Results"] = (
    model_comparison_formatted["MSE Results"].map(lambda x: f"{x:,.2f}")
)

model_comparison_formatted["RMSE Results"] = (
    model_comparison_formatted["RMSE Results"].map(lambda x: f"{x:,.2f}")
)

model_comparison_formatted["R² Score Results"] = (
    model_comparison_formatted["R² Score Results"].map(lambda x: f"{x:.4f}")
)

# Keep only the formatted columns
model_comparison_formatted = model_comparison_formatted[
    ["Model", "MAE Results", "MSE Results", "RMSE Results", "R² Score Results"]
]

model_comparison_formatted

,Model,MAE Results,MSE Results,RMSE Results,R² Score Results
0,Linear Regression,"970,043.40","1,754,318,687,330.66","1,324,506.96",0.6529
1,Decision Tree,"1,195,266.06","2,642,802,637,614.68","1,625,669.90",0.4771
2,Random Forest,"1,022,560.05","1,964,193,399,645.33","1,401,496.84",0.6114
3,Gradient Boosting,"960,578.78","1,689,379,037,287.40","1,299,761.15",0.6658
4,Tuned Gradient Boosting,"1,021,988.20","1,937,963,730,283.18","1,392,107.66",0.6166


## Best-Performing Model

Based on the test-set evaluation, the Gradient Boosting Regressor was the best-performing model. It achieved the lowest RMSE of approximately ₦1.30 million and the highest R² score of 0.6658.

The model explains approximately 66.6% of the variation in house prices and produced the lowest prediction errors among the models evaluated. Although the tuned Gradient Boosting model was optimized using GridSearchCV, it performed worse on the test set than the original Gradient Boosting model. Therefore, the original Gradient Boosting configuration was selected as the final model.